# Week 2: Frame Your Lane as an ML Task

## 1. My Lane as an ML Task

**Lane:** Lane 1 - Content Decline Prediction

**ML Task Type:** Binary Classification

**Why Classification?**
- We want to predict a binary outcome: will this page decline or not?
- Each page gets a label: `declining` (1) or `not declining` (0)
- The output is a probability score (0-1) that can be thresholded to make predictions

**Alternatives Considered:**
- **Regression:** Could predict the exact decline percentage, but editors need a yes/no decision for which pages to refresh
- **Ranking:** Could rank pages by decline risk, but the business action is binary (refresh or don't)
- **Binary Classification** is best because it matches the decision: add to refresh queue or not

---

## 2. Target or Proxy

**Target Variable:** `is_declining_label`

**Definition:** A binary label where:
- `1` = The page is declining (trend_direction == "down")
- `0` = The page is not declining (trend_direction != "down")

**Why This Target:**
- The `trend_direction` column indicates whether engagement is going up or down
- This is derived from historical engagement data over time
- It's a **proxy** for future decline because:
  - Past decline often predicts future decline
  - Editors want to catch decline early, not after it happens
  - The label is already in the data (no manual labeling needed)

**Important Note:**
- The label is derived from `trend_direction`, so `trend_direction` and `trend_pct` **CANNOT** be used as features
- This prevents data leakage (the model would just learn the rule "if trend_direction == down then predict 1")

---

## 3. Success Metric

**Primary Metric:** Precision@50

**What This Means:**
- Editors only have time to review the top 50 pages each week
- Precision@50 measures: "Of the 50 pages the model predicts as declining, how many actually decline?"
- Formula: `Precision@50 = (True Positives in top 50) / 50`

**Why This Metric:**
- Matches the business decision (editors take action on the top 50)
- The baseline (hand-rule) gets Precision@50 = 0.240
- ML model can achieve Precision@50 = ~0.740 (3x improvement)

**Secondary Metrics:**
- **Precision@100:** If editors can review 100 pages
- **Recall:** To ensure we're not missing too many declining pages
- **AUC-ROC:** To evaluate overall model performance

---

## 4. Unit of Analysis

### One Row = One Page

In our dataset, each row represents a **single page/content item** with its historical engagement data and features.

In [ ]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Show the unit of analysis as a real dataframe
print("=" * 60)
print("UNIT OF ANALYSIS: One row = One page")
print("=" * 60)
print(f"Total pages: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print("\nFirst 5 rows (sample of pages):")

# Show first 5 rows with key columns
display_cols = ['url', 'trend_direction', 'views_7d', 'views_28d', 'content_type', 'word_count']
display_cols = [col for col in display_cols if col in df.columns]
print(df[display_cols].head())

print("\n" + "=" * 60)
print("TARGET COLUMN (What we're predicting):")
print("=" * 60)

# Show the target column
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(df[['url', 'trend_direction', 'is_declining_label']].head())

print("\n" + "=" * 60)
print("TARGET DISTRIBUTION:")
print("=" * 60)

# Show class balance
decline_counts = df['is_declining_label'].value_counts()
decline_pct = df['is_declining_label'].mean() * 100
print(f"Not declining: {decline_counts[0]:,} rows ({100-decline_pct:.1f}%)")
print(f"Declining:     {decline_counts[1]:,} rows ({decline_pct:.1f}%)")
print(f"\nClass imbalance: {decline_pct:.1f}% declining")

## 5. Why ML Beats a Fixed Rule

### The Fixed Rule Approach

A fixed rule might be:
> "If page views drop by more than 10% in the last 7 days, mark as declining."

**Problems with Fixed Rules:**
1. **Too Simplistic:** Decline depends on many factors, not just one metric
2. **No Adaptation:** Rules can't learn from new patterns
3. **Misses Complex Patterns:** A page might decline even without a sharp drop
4. **High False Alarms:** Rigid thresholds cause many false positives

### Why ML Is Better

ML can find patterns that fixed rules miss:

| Factor | How ML Handles It |
|--------|-------------------|
| **Multiple features** | Combines 35+ signals (page age, content type, author, historical patterns) |
| **Non-linear patterns** | Captures interactions between features (e.g., old content + low engagement + specific type) |
| **Self-improving** | Learns from new data and adapts over time |
| **Probability-based** | Gives a confidence score, not just a yes/no |
| **Performance** | 3x better Precision@50 than hand-rule (0.740 vs 0.240) |

### The Evidence

**Baseline (hand-rule):** Precision@50 = 0.240
- Editors review 50 pages
- Only ~12 of them actually decline (50 × 0.240)
- **38 pages are wasted effort**

**ML Model:** Precision@50 = 0.740
- Editors review 50 pages
- ~37 of them actually decline (50 × 0.740)
- **Only 13 pages are wasted effort**

**Result:** ML reduces wasted editorial time by 66%!

---

## 6. Self-Check

### ML Task Type Checklist

| Question | Answer | Status |
|----------|--------|--------|
| Is this a classification, ranking, or scoring problem? | Binary Classification | ✅ |
| What is the target variable? | `is_declining_label` (1 if declining, 0 if not) | ✅ |
| What metric captures "good"? | Precision@50 | ✅ |
| What is the unit of analysis? | One page | ✅ |
| Why does ML beat a fixed rule? | 3x better precision, captures complex patterns | ✅ |
| Does the output support a real action? | Yes - editors refresh the top 50 predicted pages | ✅ |

### The ML Loop for My Lane
